<a href="https://colab.research.google.com/github/jonatanban/Attitude-Estimation-by-LSTM-network-/blob/main/Atittude_Estimation_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Data Generation

The goal of the `generate_dataset` function is to create examples of "sensor noise". This helps train the network to find the constant IMU errors when the sensor is not moving (static).

* **The Constant Error (Target / Ground Truth):** For each trajectory, we generate a random Bias for the accelerometer and a Drift for the gyroscope:

$$\mathbf{b}_a \sim \mathcal{N}(-10, 10) \times 10^{-3} \times g$$


$$\mathbf{b}_g \sim \mathcal{N}(-0.1, 0.1) \times \frac{\pi}{180 \times 3600}$$



These values are combined into the target vector that the network needs to predict: $\mathbf{y} = [\mathbf{b}_a, \mathbf{b}_g]^T \in \mathbb{R}^6$.
* **Adding White Noise (Sensor Noise):** We simulate random noise that changes based on the sample rate ($dt = 0.005$):

$$\mathbf{n}_a(t) \sim \mathcal{N}(0, (0.2 \sqrt{dt})^2), \quad \mathbf{n}_g(t) \sim \mathcal{N}(0, (0.02 \frac{\pi}{180 \times 60} \sqrt{dt})^2)$$



The raw measurement at any time $t$ is the sum of the constant error and the random noise:

$$\mathbf{x}_t = \mathbf{b} + \mathbf{n}(t)$$


* **Sliding Windows:** We pack the measurements into continuous time windows of 50 samples. This is the input for the model:

$$\mathbf{X}_i = [\mathbf{x}_{i}, \mathbf{x}_{i+1}, \dots, \mathbf{x}_{i+49}]^T \in \mathbb{R}^{50 \times 6}$$




---

### 2. Model Architecture

The `IMUNoiseEstimator` model is designed to extract the static error from the noisy time series:

* **LSTM Layer:** A network with 2 layers and 64 hidden neurons. It scans the time window $\mathbf{X}_i$, remembers the history, and learns to ignore the random noise.
* **Linear Layer (FC - Fully Connected):** It takes the output from the last step of the LSTM and reduces it to 6 values. This is the model's final prediction for the bias and drift:

$$\hat{\mathbf{y}} = \text{FC}(\text{LSTM}(\mathbf{X}_i)) \in \mathbb{R}^6$$




---

### 3. Training Process

* **Loss Function:** We use Mean Squared Error (MSE Loss) to measure the distance between the model's prediction and the real answer:

$$\mathcal{L} = \frac{1}{M} \sum_{k=1}^{M} \|\hat{\mathbf{y}}_k - \mathbf{y}_k\|^2$$


* **Running Results:** The optimizer (Adam) updated the network's weights and successfully dropped the Loss from $O ( 10^{-4}$) in the first epoch down to $O( 10^{-7}$) in the tenth epoch. This shows that the model learned very well. The trained weights were successfully saved to the `trained_imu_lstm.pth` file.

In [2]:
# CELL 1 - Data generation and LSTM training
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import time

# Use GPU/MPS if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Running on device: {device}")
torch.set_default_dtype(torch.float32)


class IMUNoiseEstimator(nn.Module):
    """LSTM that maps a 50-sample IMU window to the constant bias/drift (6 values)."""
    def __init__(self, input_dim=6, hidden_dim=64, num_layers=2):
        super(IMUNoiseEstimator, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 6)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        return self.fc(lstm_out[:, -1, :])   # use the last time step only


def generate_dataset(num_traj=150, samples=2000, win=50):
    """Create windows of (constant bias/drift + white noise). Target = the constant."""
    print(f"Generating {num_traj} training trajectories...")
    X, Y = [], []
    dt = 0.005

    acc_noise_std = 0.02    # accelerometer white-noise std [m/s^2]
    gyro_noise_std = 0.002  # gyroscope white-noise std [rad/s]

    for _ in range(num_traj):
        # One constant bias/drift per trajectory, drawn from a zero-mean Gaussian
        true_acc_bias = np.random.normal(0.0, 0.1, 3).astype(np.float32)
        true_gyro_drift = np.random.normal(0.0, 0.01, 3).astype(np.float32)
        target = np.concatenate([true_acc_bias, true_gyro_drift]).astype(np.float32)

        # White noise, scaled by sqrt(dt) (discrete-time noise)
        acc_noise = acc_noise_std * np.random.randn(samples, 3).astype(np.float32) * np.sqrt(dt)
        gyro_noise = gyro_noise_std * np.random.randn(samples, 3).astype(np.float32) * np.sqrt(dt)

        # "Gravity-blind": the network sees only the error signal (no physics)
        imu_raw = np.concatenate([true_acc_bias + acc_noise, true_gyro_drift + gyro_noise], axis=1)

        for i in range(samples - win):        # sliding windows, stride 1
            X.append(imu_raw[i:i + win, :])
            Y.append(target)

    np.savez("imu_dataset.npz", X=np.array(X), Y=np.array(Y))
    print("Data saved to imu_dataset.npz\n")

def train(epochs=10):
    print("Loading data...")
    data = np.load("imu_dataset.npz")
    X_tensor = torch.tensor(data['X'], dtype=torch.float32).to(device)
    Y_tensor = torch.tensor(data['Y'], dtype=torch.float32).to(device)
    loader = DataLoader(TensorDataset(X_tensor, Y_tensor), batch_size=256, shuffle=True)

    model = IMUNoiseEstimator().to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.002)
    criterion = nn.MSELoss()

    print("Training...")
    start_time = time.time()
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for bx, by in loader:
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch + 1}/{epochs} - Loss: {total_loss / len(loader):.2e}")

    torch.save(model.cpu().state_dict(), "trained_imu_lstm.pth")
    print(f"\nDone in {time.time() - start_time:.1f} s. Weights saved to trained_imu_lstm.pth")


if __name__ == "__main__":
    generate_dataset()
    train()


Running on device: cuda
Generating 150 training trajectories...
Data saved to imu_dataset.npz

Loading data...
Training...
Epoch 1/10 - Loss: 1.43e-04
Epoch 2/10 - Loss: 9.73e-07
Epoch 3/10 - Loss: 8.07e-07
Epoch 4/10 - Loss: 7.53e-07
Epoch 5/10 - Loss: 6.89e-07
Epoch 6/10 - Loss: 6.42e-07
Epoch 7/10 - Loss: 5.72e-07
Epoch 8/10 - Loss: 6.09e-07
Epoch 9/10 - Loss: 5.16e-07
Epoch 10/10 - Loss: 4.65e-07

Done in 50.9 s. Weights saved to trained_imu_lstm.pth


Detailed Navigation & Filter Explanation

This code implements a high-precision **Inertial Navigation System (INS)** integrated with an **Error-State Kalman Filter (ESKF)**.

#### 1. Reference Frames & Gravity Model

The system operates in the **ECEF** (Earth-Centered, Earth-Fixed) frame.

* **Coordinate Transformation:** `nav_ecef_to_geod` converts Cartesian coordinates $(X, Y, Z)$ to Geodetic $(lat, lon, alt)$. This is necessary to find the local **NED** (North-East-Down) horizon.
* **Gravity Model:** Instead of a simple constant, `nav_pos_to_grav` uses a spherical harmonic model (J2-J4 coefficients). This accounts for the Earth's irregular shape:

$$\vec{g}_{eff} = \text{Gravity} + \text{Centrifugal Force}$$



#### 2. INS Mechanics (Propagation)

The function `nav_inertial_prop` performs the numerical integration of Newton’s equations of motion.

* **Velocity Update:** The measured acceleration from the IMU is rotated into the ECEF frame and corrected for gravity and the **Coriolis effect**:

$$\vec{V}_{k+1} = \vec{V}_k + \Delta \vec{V}_{rotated} + (\vec{g} - 2\vec{\omega}_{ie} \times \vec{V}_k) \Delta t$$


* **Orientation (Quaternions):** Orientation is tracked using 4-element quaternions to ensure mathematical stability during high-angle maneuvers.

#### 3. Error-State Kalman Filter (ESKF)

The ESKF tracks the **uncertainty** of the system. We use a **15-state vector** $\delta \mathbf{x}$:

* $\delta P, \delta V$ (Position and Velocity errors)
* $\vec{\psi}$ (Attitude/Tilt errors)
* $\delta b_a, \delta b_g$ (Accelerometer and Gyroscope Biases)

**Key Functions:**

* **`nav_dyn_mat`:** This builds the state transition matrix. It defines how a tilt error $\psi$ "leaks" into the velocity error through the acceleration vector.
* **`nav_zupt_update` (Zero Velocity Update):** This is a critical correction step. When the vehicle is stationary, the "observed" velocity should be zero. The filter uses the difference between the INS velocity and zero to correct the **Azimuth** and sensor biases:

$$\text{Innovation (z)} = 0 - \vec{V}_{INS}$$


* **`nav_correct`:** This function takes the estimated errors from the Kalman filter and applies them back to the "True" navigation state to keep the path accurate.

#### 4. Covariance & Noise ($P$ and $Q$)

* **$P$ Matrix:** Represents our confidence. It starts large (uncertain) and shrinks as we get more ZUPT updates.
* **$Q$ Matrix:** Represents the noise of the IMU sensors (Random Walk). It tells the filter how much the error grows every second.

In [3]:
# CELL 2 - Navigation engine (INS mechanization + Error-State Kalman Filter)
import numpy as np
import torch
import torch.nn as nn
from scipy.linalg import sqrtm
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
torch.set_default_dtype(torch.float64)   # double precision for the physics

dt = 0.005
w_ie_e = np.array([0, 0, 7.2921151467e-5], dtype=np.float64)   # Earth rotation rate [rad/s]
GM = 3.986004418e14
gravity = 9.80665


def nav_skew(vec):
    """Skew-symmetric matrix of a 3-vector (cross-product operator)."""
    vec = np.asarray(vec).reshape(-1)
    return np.array([[0, -vec[2], vec[1]], [vec[2], 0, -vec[0]], [-vec[1], vec[0], 0]])


def nav_quat_to_dcm(q):
    """Quaternion -> direction cosine matrix."""
    dcm = np.zeros((3, 3), dtype=np.float64)
    dcm[0, 0] = q[0]**2 + q[1]**2 - q[2]**2 - q[3]**2
    dcm[0, 1] = 2 * (q[1] * q[2] - q[0] * q[3])
    dcm[0, 2] = 2 * (q[1] * q[3] + q[0] * q[2])
    dcm[1, 0] = 2 * (q[1] * q[2] + q[0] * q[3])
    dcm[1, 1] = q[0]**2 - q[1]**2 + q[2]**2 - q[3]**2
    dcm[1, 2] = 2 * (q[2] * q[3] - q[0] * q[1])
    dcm[2, 0] = 2 * (q[1] * q[3] - q[0] * q[2])
    dcm[2, 1] = 2 * (q[2] * q[3] + q[0] * q[1])
    dcm[2, 2] = q[0]**2 - q[1]**2 - q[2]**2 + q[3]**2
    return dcm


def nav_dcm_to_quat(dcm):
    """Direction cosine matrix -> quaternion (numerically stable branch selection)."""
    q = np.zeros(4)
    tr = np.trace(dcm)
    if tr > 0:
        S = np.sqrt(tr + 1.0) * 2
        q[0], q[1], q[2], q[3] = 0.25 * S, (dcm[2,1] - dcm[1,2]) / S, (dcm[0,2] - dcm[2,0]) / S, (dcm[1,0] - dcm[0,1]) / S
    elif (dcm[0,0] > dcm[1,1]) and (dcm[0,0] > dcm[2,2]):
        S = np.sqrt(1.0 + dcm[0,0] - dcm[1,1] - dcm[2,2]) * 2
        q[0], q[1], q[2], q[3] = (dcm[2,1] - dcm[1,2]) / S, 0.25 * S, (dcm[0,1] + dcm[1,0]) / S, (dcm[0,2] + dcm[2,0]) / S
    elif dcm[1,1] > dcm[2,2]:
        S = np.sqrt(1.0 + dcm[1,1] - dcm[0,0] - dcm[2,2]) * 2
        q[0], q[1], q[2], q[3] = (dcm[0,2] - dcm[2,0]) / S, (dcm[0,1] + dcm[1,0]) / S, 0.25 * S, (dcm[1,2] + dcm[2,1]) / S
    else:
        S = np.sqrt(1.0 + dcm[2,2] - dcm[0,0] - dcm[1,1]) * 2
        q[0], q[1], q[2], q[3] = (dcm[1,0] - dcm[0,1]) / S, (dcm[0,2] + dcm[2,0]) / S, (dcm[1,2] + dcm[2,1]) / S, 0.25 * S
    return q


def nav_angles_to_dcm(angles):
    """Euler angles (roll, pitch, yaw) -> direction cosine matrix."""
    phi, theta, psi = angles
    PHI = np.array([[1, 0, 0], [0, np.cos(phi), np.sin(phi)], [0, -np.sin(phi), np.cos(phi)]])
    THETA = np.array([[np.cos(theta), 0, -np.sin(theta)], [0, 1, 0], [np.sin(theta), 0, np.cos(theta)]])
    PSI = np.array([[np.cos(psi), np.sin(psi), 0], [-np.sin(psi), np.cos(psi), 0], [0, 0, 1]])
    return PHI @ THETA @ PSI


def nav_ecef_to_geod(pe, tolerance=1e-10):
    """ECEF position -> geodetic latitude, longitude, altitude (iterative)."""
    a, f = 6378137.0, 1 / 298.257223563
    e2 = f * (2 - f)
    x, y, z = pe
    p = np.sqrt(x**2 + y**2)
    lon = np.arctan2(y, x)
    lat, alt, N = np.arctan2(z, p * (1 - e2)), 0, a
    for _ in range(10):
        lat_old = lat
        N = a / np.sqrt(1 - e2 * np.sin(lat)**2)
        alt = (p / np.cos(lat)) - N
        lat = np.arctan2(z, p * (1 - e2 * (N / (N + alt))))
        if abs(lat - lat_old) < tolerance:
            break
    return lat, lon, alt


def nav_c_ecef_to_ned(pe):
    """Rotation matrix from ECEF to the local North-East-Down frame."""
    lat, lon, _ = nav_ecef_to_geod(pe)
    PERM = np.array([[0, 0, 1], [0, 1, 0], [-1, 0, 0]])
    LAT = np.array([[np.cos(-lat), 0, -np.sin(-lat)], [0, 1, 0], [np.sin(-lat), 0, np.cos(-lat)]])
    LON = np.array([[1, 0, 0], [0, np.cos(lon), np.sin(lon)], [0, -np.sin(lon), np.cos(lon)]])
    return LAT @ LON @ PERM


def nav_angles_to_quat(pos, angles):
    """Initial attitude (Euler angles in NED) -> body-to-ECEF quaternion."""
    return nav_dcm_to_quat((nav_angles_to_dcm(angles) @ nav_c_ecef_to_ned(pos)).T)


def nav_pos_to_grav(pe):
    """Gravity vector in ECEF using J2-J4 spherical-harmonic corrections."""
    x, y, z = pe
    r = np.linalg.norm(pe)
    j2, j3, j4 = 6.6063107600492e10, -3.28504625162162e14, -3.33255700733655e20
    g0 = GM / r**2
    h11 = (j2 / r**2) * (1 - 5 * (z / r)**2)
    h12 = -5 * j3 * (z / r**4) * (-3 + 7 * (z / r)**2)
    h13 = 5 * j4 * (1 / r**4) * (-3 + 42 * (z / r)**2 - 63 * (z / r)**4)
    h21 = ((z * j2) / r**3) * (3 - 5 * (z / r)**2)
    h22 = (j3 / r**3) * (30 * (z / r)**2 - 35 * (z / r)**4 - 3)
    h23 = ((5 * z * j4) / r**5) * (70 * (z / r)**2 - 63 * (z / r)**4 - 15)
    return np.array([-g0 * (x/r) * (1 + h11 + h12 + h13),
                     -g0 * (y/r) * (1 + h11 + h12 + h13),
                     -g0 * (z/r) - g0 * (h21 + h22 + h23)])


def nav_static_to_inertial(pos, angles):
    """Ideal specific force and angular rate a perfect static IMU would measure."""
    c_n_b = nav_angles_to_dcm(angles)
    c_e_n = nav_c_ecef_to_ned(pos)
    return (c_n_b @ c_e_n @ (-nav_pos_to_grav(pos) + np.cross(w_ie_e, np.cross(w_ie_e, pos))),
            c_n_b @ c_e_n @ w_ie_e)


def nav_quat_prop(q0, dthet, omegaInc):
    """Propagate the quaternion by a body-rate increment and Earth-rate increment."""
    sx, sy = np.linalg.norm(dthet) / 2, np.linalg.norm(omegaInc) / 2
    wx, wy, wz = dthet
    X = 0.5 * np.array([[0, -wx, -wy, -wz], [wx, 0, wz, -wy], [wy, -wz, 0, wx], [wz, wy, -wx, 0]])
    Y = 0.5 * np.array([[0, 0, 0, omegaInc[2]], [0, 0, omegaInc[2], 0], [0, -omegaInc[2], 0, 0], [-omegaInc[2], 0, 0, 0]])
    XX = X + np.eye(4) if sx < 1e-16 else np.sin(sx) / sx * X + np.cos(sx) * np.eye(4)
    YY = Y + np.eye(4) if sy < 1e-16 else np.sin(sy) / sy * Y + np.cos(sy) * np.eye(4)
    q1 = XX @ YY @ q0
    return q1 / np.linalg.norm(q1)


def nav_inertial_prop(p_old, v_old, q_old, dvel, dthet, dt):
    """One INS step: update attitude, then velocity (gravity + Coriolis + centrifugal), then position."""
    q_new = nav_quat_prop(q_old, dthet, w_ie_e * dt)
    v_new = (v_old + nav_quat_to_dcm(q_new) @ dvel + nav_pos_to_grav(p_old) * dt
             - np.cross(w_ie_e, np.cross(w_ie_e, p_old)) * dt
             - 2 * np.cross(w_ie_e, v_old) * dt)
    p_new = p_old + (v_old + v_new) * dt / 2
    return p_new, v_new, q_new


def nav_dyn_mat(pos, dvel, q, dt):
    """15x15 error-state transition matrix F (position, velocity, tilt, accel bias, gyro drift)."""
    c_b_e = nav_quat_to_dcm(q)
    acc_e = (c_b_e @ dvel) / dt
    x, y, z = pos
    r = np.linalg.norm(pos)
    r_cubic, r_fifth = r**3, r**5
    grad_g = -GM * np.array([
        [1/r_cubic - 3*x**2/r_fifth, -3*x*y/r_fifth, -3*x*z/r_fifth],
        [-3*y*x/r_fifth, 1/r_cubic - 3*y**2/r_fifth, -3*y*z/r_fifth],
        [-3*z*x/r_fifth, -3*z*y/r_fifth, 1/r_cubic - 3*z**2/r_fifth]
    ])

    dyn_mat = np.zeros((15, 15))
    dyn_mat[0:3, 3:6] = np.eye(3)                                  # dPos/dt = Vel
    dyn_mat[3:6, 0:3] = grad_g - nav_skew(w_ie_e) @ nav_skew(w_ie_e)
    dyn_mat[3:6, 3:6] = -2 * nav_skew(w_ie_e)                      # Coriolis
    dyn_mat[3:6, 6:9] = -nav_skew(acc_e)                           # tilt -> velocity
    dyn_mat[3:6, 9:12] = -c_b_e                                    # accel bias -> velocity
    dyn_mat[6:9, 6:9] = -nav_skew(w_ie_e)
    dyn_mat[6:9, 12:15] = -c_b_e                                   # gyro drift -> tilt
    return dyn_mat


def nav_kal_prop(x_hat, dyn_mat, P, Q, dt):
    """Kalman predict step: propagate error state and covariance."""
    Phi = np.eye(15) + dyn_mat * dt
    x_hat = Phi @ x_hat
    P = Phi @ P @ Phi.T + Q * dt
    return x_hat, P


def nav_zupt_update(x_hat, P, vel, R_zupt):
    """Zero-Velocity Update: the true velocity is 0, so the innovation is z = 0 - v_INS."""
    H = np.zeros((3, 15))
    H[0:3, 3:6] = np.eye(3)
    z = -vel.reshape(3, 1)

    S = H @ P @ H.T + R_zupt
    K = P @ H.T @ np.linalg.inv(S)
    x_hat = x_hat + K @ (z - H @ x_hat)
    P = (np.eye(15) - K @ H) @ P @ (np.eye(15) - K @ H).T + K @ R_zupt @ K.T   # Joseph form
    return x_hat, P


def nav_correct(pos, vel, quat, ba, bg, x_hat):
    """Closed-loop correction: feed the estimated error back into the nominal state."""
    dpos, dvel, tilt = x_hat[0:3], x_hat[3:6], x_hat[6:9]
    dba, dbg = x_hat[9:12], x_hat[12:15]
    pos_new = pos + dpos
    vel_new = vel + dvel
    quat_new = nav_dcm_to_quat((np.eye(3) - nav_skew(tilt)) @ nav_quat_to_dcm(quat))
    ba_new = ba + dba
    bg_new = bg + dbg
    return pos_new, vel_new, quat_new, ba_new, bg_new


def nav_angles_ned_to_tilt_ecef_cov(pos, angles_cov, angles):
    """Convert an attitude-angle covariance (NED) into a tilt covariance (ECEF)."""
    theta, psi = angles[1], angles[2]
    mat_angles_to_tilt = np.array([
        [-np.cos(theta)*np.cos(psi), np.sin(psi), 0],
        [-np.cos(theta)*np.sin(psi), -np.cos(psi), 0],
        [np.sin(theta), 0, -1]
    ])
    tilt_ned_cov = mat_angles_to_tilt @ angles_cov @ mat_angles_to_tilt.T
    c_e_n = nav_c_ecef_to_ned(pos)
    return c_e_n.T @ tilt_ned_cov @ c_e_n


def nav_kal_init(q, pos):
    """Initialize the error state (0), the covariance P, and the process noise Q."""
    x_hat = np.zeros((15, 1))
    P = np.zeros((15, 15))
    P[0:3, 0:3] = np.diag(np.array([10.0, 10.0, 10.0])) ** 2      # position uncertainty
    P[3:6, 3:6] = np.diag(np.array([0.5, 0.5, 0.5])) ** 2         # velocity uncertainty

    init_angles = np.array([22.0, 33.0, 44.0]) * np.pi / 180
    angles_cov = np.diag(np.array([17.0, 17.0, 17.0]) / 1000) ** 2
    P[6:9, 6:9] = nav_angles_ned_to_tilt_ecef_cov(pos, angles_cov, init_angles)
    P[9:12, 9:12] = np.eye(3) * (5 * 1e-3 * gravity) ** 2         # accel bias uncertainty
    P[12:15, 12:15] = np.eye(3) * (0.02 * np.pi/3600/180) ** 2    # gyro drift uncertainty

    c_b_e = nav_quat_to_dcm(q)
    Q = np.zeros((15, 15))
    Q[3:6, 3:6] = c_b_e @ (np.eye(3) * 0.1**2) @ c_b_e.T                       # velocity process noise
    Q[6:9, 6:9] = c_b_e @ (np.eye(3) * (0.01 * np.pi/60/180)**2) @ c_b_e.T     # tilt process noise
    return x_hat, P, Q


print("Cell 2 done: navigation engine loaded.")



Cell 2 done: navigation engine loaded.


### Part 3: Final Validation and A/B Test

In this part, we run a "competition" between two methods to see which one estimates the azimuth better.

#### 1. Simulation Setup

We run a 15-second simulation where the vehicle is standing still (static). To test the systems, we add heavy errors to the raw data:

* **Constant Errors:** Bias in the accelerometer and Drift in the gyroscope.
* **Random Noise:** White noise to simulate environment interference.

#### 2. The Competitors

The code runs two separate navigation paths at the same time:

* **Classic Model (Classic ESKF):** Uses the noisy data directly inside the Kalman Filter.
* **Hybrid Model (Hybrid AI-ESKF):** Uses our trained LSTM network to "clean" the data before it enters the navigation equations.

#### 3. AI Integration

The Hybrid system keeps a "Buffer" of the last 50 measurements. In every step, the LSTM predicts the bias, and the code corrects the measurement:


$$\mathbf{u}_{corrected} = \mathbf{u}_{noisy} - \text{AI\_Prediction}$$

#### 4. ZUPT Updates

Every 0.5 seconds (100 steps), both systems perform a **ZUPT** (Zero Velocity Update). This is where the Kalman Filter uses the fact that the vehicle is static to fix the **Azimuth** (heading) error.

#### 5. Results (Plotting)

The final graphs show the position errors (X, Y, Z) and the Azimuth Uncertainty. You can clearly see that the Green line (Hybrid Model) is much more stable and accurate than the Red line (Classic Model).

**Conclusion:** Using Deep Learning to clean sensor data (Model-Based Deep Learning) makes the physical navigation system much stronger.

In [4]:
# CELL 3 - A/B test: Classic ESKF vs Hybrid (LSTM + ESKF), and plotting
import numpy as np
import torch
import torch.nn as nn
import plotly.graph_objects as go
from plotly.subplots import make_subplots


class IMUNoiseEstimator(nn.Module):
    """Same network as Cell 1, in double precision to match the physics."""
    def __init__(self, input_dim=6, hidden_dim=64, num_layers=2):
        super(IMUNoiseEstimator, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dtype=torch.float64)
        self.fc = nn.Linear(hidden_dim, 6, dtype=torch.float64)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        return self.fc(lstm_out[:, -1, :])


def get_azimuth(pos, quat):
    """Heading (0-360 deg) from the body-to-NED matrix."""
    c_b_n = nav_c_ecef_to_ned(pos) @ nav_quat_to_dcm(quat)
    az = np.arctan2(c_b_n[1, 0], c_b_n[0, 0]) * 180.0 / np.pi
    return az if az >= 0 else az + 360.0


def run_ab_test_with_zupt():
    """Run 15 s static alignment for the Classic and Hybrid filters and log all signals."""
    dt = 0.005                                # 200 Hz
    rows = int(np.floor(200 * 60 * 0.25))     # 15 seconds
    init_pos = np.array([6378137.0, 0.0, 0.0])
    init_angles = np.array([22.0, 33.0, 44.0]) * np.pi / 180
    true_azimuth_deg = init_angles[2] * 180.0 / np.pi

    acc_ideal, pqr_ideal = nav_static_to_inertial(init_pos, init_angles)

    test_bias = np.array([0.08, -0.05, 0.04])      # true accelerometer bias [m/s^2]
    test_drift = np.array([0.002, 0.003, -0.004])  # true gyroscope drift [rad/s]

    # White noise scaled by sqrt(dt) to match the training data
    acc_noisy_rate = acc_ideal + test_bias + 0.02 * np.sqrt(dt) * np.random.randn(rows, 3)
    gyro_noisy_rate = pqr_ideal + test_drift + 0.002 * np.sqrt(dt) * np.random.randn(rows, 3)

    dvel_noisy = acc_noisy_rate * dt
    dthet_noisy = gyro_noisy_rate * dt

    # Load the trained network
    ai_model = IMUNoiseEstimator().double()
    try:
        ai_model.load_state_dict(torch.load("trained_imu_lstm.pth", map_location='cpu', weights_only=True))
        print("LSTM weights loaded.")
    except Exception as e:
        print(f"Warning: running with random weights. Error: {e}")
    ai_model.eval()

    # Classic filter state
    pos_c, vel_c = init_pos.copy(), np.zeros(3)
    quat_c = nav_angles_to_quat(init_pos, init_angles)
    ba_c, bg_c = np.zeros(3), np.zeros(3)
    x_hat_c, P_c, Q_c = nav_kal_init(quat_c, init_pos)

    # Hybrid filter state
    pos_h, vel_h = init_pos.copy(), np.zeros(3)
    quat_h = nav_angles_to_quat(init_pos, init_angles)
    ba_h, bg_h = np.zeros(3), np.zeros(3)
    x_hat_h, P_h, Q_h = nav_kal_init(quat_h, init_pos)

    R_zupt = np.eye(3) * (0.01) ** 2

    # Logs
    out_pos_c, out_pos_h = np.zeros((rows, 3)), np.zeros((rows, 3))
    out_az_err_c, out_az_err_h = np.zeros(rows), np.zeros(rows)
    out_ba_c, out_bg_c = np.zeros((rows, 3)), np.zeros((rows, 3))
    out_ba_h, out_bg_h = np.zeros((rows, 3)), np.zeros((rows, 3))
    out_nn_ba, out_nn_bg = np.zeros((rows, 3)), np.zeros((rows, 3))   # raw LSTM output

    imu_buffer = []
    print("Running Classic vs Hybrid comparison (with ZUPT)...")
    for i in range(rows - 1):
        # Residual = measurement - ideal (this is bias/drift + noise), the LSTM input
        acc_residual = acc_noisy_rate[i] - acc_ideal
        gyro_residual = gyro_noisy_rate[i] - pqr_ideal
        imu_buffer.append(np.concatenate([acc_residual, gyro_residual]))
        if len(imu_buffer) > 50:
            imu_buffer.pop(0)

        # LSTM prediction once the 50-sample window is full
        if len(imu_buffer) == 50:
            with torch.no_grad():
                pred = ai_model(torch.tensor(np.array([imu_buffer]), dtype=torch.float64)).numpy().flatten()
            pred_ba, pred_bg = pred[0:3], pred[3:6]
        else:
            pred_ba, pred_bg = np.zeros(3), np.zeros(3)
        out_nn_ba[i], out_nn_bg[i] = pred_ba, pred_bg

        # ----------------- CLASSIC: raw IMU into the filter -----------------
        dvel_c_in = dvel_noisy[i] - (ba_c * dt)
        dthet_c_in = dthet_noisy[i] - (bg_c * dt)
        pos_c, vel_c, quat_c = nav_inertial_prop(pos_c, vel_c, quat_c, dvel_c_in, dthet_c_in, dt)
        x_hat_c, P_c = nav_kal_prop(x_hat_c, nav_dyn_mat(pos_c, dvel_c_in, quat_c, dt), P_c, Q_c, dt)
        if i % 100 == 0 and i > 0:                 # ZUPT every 0.5 s
            x_hat_c, P_c = nav_zupt_update(x_hat_c, P_c, vel_c, R_zupt)
            pos_c, vel_c, quat_c, ba_c, bg_c = nav_correct(pos_c, vel_c, quat_c, ba_c, bg_c, x_hat_c.flatten())
            x_hat_c.fill(0)
        out_pos_c[i], out_ba_c[i], out_bg_c[i] = pos_c, ba_c, bg_c
        az_err_c = get_azimuth(pos_c, quat_c) - true_azimuth_deg
        out_az_err_c[i] = (az_err_c + 180) % 360 - 180   # wrap to [-180, 180]

        # ----------------- HYBRID: subtract LSTM estimate first -----------------
        dvel_h_in = dvel_noisy[i] - (pred_ba * dt) - (ba_h * dt)
        dthet_h_in = dthet_noisy[i] - (pred_bg * dt) - (bg_h * dt)
        pos_h, vel_h, quat_h = nav_inertial_prop(pos_h, vel_h, quat_h, dvel_h_in, dthet_h_in, dt)
        x_hat_h, P_h = nav_kal_prop(x_hat_h, nav_dyn_mat(pos_h, dvel_h_in, quat_h, dt), P_h, Q_h, dt)
        if i % 100 == 0 and i > 0:                 # ZUPT every 0.5 s
            x_hat_h, P_h = nav_zupt_update(x_hat_h, P_h, vel_h, R_zupt)
            pos_h, vel_h, quat_h, ba_h, bg_h = nav_correct(pos_h, vel_h, quat_h, ba_h, bg_h, x_hat_h.flatten())
            x_hat_h.fill(0)
        out_pos_h[i] = pos_h
        out_ba_h[i] = ba_h + pred_ba               # total hybrid estimate (LSTM + Kalman)
        out_bg_h[i] = bg_h + pred_bg
        az_err_h = get_azimuth(pos_h, quat_h) - true_azimuth_deg
        out_az_err_h[i] = (az_err_h + 180) % 360 - 180

    # Fill the last sample
    out_ba_c[-1], out_bg_c[-1] = ba_c, bg_c
    out_ba_h[-1], out_bg_h[-1] = ba_h + pred_ba, bg_h + pred_bg
    out_nn_ba[-1], out_nn_bg[-1] = pred_ba, pred_bg

    return (np.arange(rows) * dt, out_pos_c, out_az_err_c, out_pos_h, out_az_err_h,
            init_pos, test_bias, test_drift, out_ba_c, out_bg_c, out_ba_h, out_bg_h,
            out_nn_ba, out_nn_bg)


def plot_final_ab_test(time, pos_c, az_err_c, pos_h, az_err_h, init_pos, true_ba, true_bg, ba_c, bg_c, ba_h, bg_h):
    """Navigation performance: position errors, azimuth error, RMSE, and total bias/drift tracking."""
    err_c = pos_c - init_pos
    err_h = pos_h - init_pos
    dist_c = np.linalg.norm(err_c, axis=1)
    dist_h = np.linalg.norm(err_h, axis=1)
    rmse_c = np.sqrt(np.cumsum(dist_c**2) / np.arange(1, len(dist_c) + 1))
    rmse_h = np.sqrt(np.cumsum(dist_h**2) / np.arange(1, len(dist_h) + 1))

    # Calculate RMS (Norm) for the true bias/drift scalars
    true_ba_norm = np.linalg.norm(true_ba)
    true_bg_norm = np.linalg.norm(true_bg)

    # Calculate RMS (Norm) over time (axis=1 computes norm across X,Y,Z for each time step)
    ba_c_norm = np.linalg.norm(ba_c[:-1], axis=1)
    ba_h_norm = np.linalg.norm(ba_h[:-1], axis=1)

    bg_c_norm = np.linalg.norm(bg_c[:-1], axis=1)
    bg_h_norm = np.linalg.norm(bg_h[:-1], axis=1)

    fig = make_subplots(
        rows=4, cols=2, shared_xaxes=True, vertical_spacing=0.06,
        subplot_titles=(
            'X Position Error [m]', 'Azimuth Error [deg]',
            'Y Position Error [m]', 'Cumulative Position RMSE [m]',
            'Z Position Error [m]', 'Accel Bias Norm (RMS): Classic vs Hybrid [m/s^2]',
            'Gyro Drift Norm (RMS): Classic vs Hybrid [rad/s]', ''
        )
    )

    # Left column: position errors on X, Y, Z
    for i, axis_name in enumerate(['X', 'Y', 'Z']):
        fig.add_trace(go.Scatter(x=time[:-1], y=err_c[:-1, i], name='Classic', line=dict(color='#ef4444', width=1.5), legendgroup="Classic", showlegend=(i == 0)), row=i+1, col=1)
        fig.add_trace(go.Scatter(x=time[:-1], y=err_h[:-1, i], name='Hybrid', line=dict(color='#10b981', width=2.5), legendgroup="Hybrid", showlegend=(i == 0)), row=i+1, col=1)
        fig.add_trace(go.Scatter(x=time[:-1], y=np.zeros_like(time[:-1]), line=dict(color='black', dash='dash'), showlegend=False), row=i+1, col=1)

    # Right column, row 1: azimuth error
    fig.add_trace(go.Scatter(x=time[:-1], y=az_err_c[:-1], line=dict(color='#ef4444', width=2), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=time[:-1], y=az_err_h[:-1], line=dict(color='#10b981', width=3), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=time[:-1], y=np.zeros_like(time[:-1]), line=dict(color='black', dash='dash'), showlegend=False), row=1, col=2)

    # Right column, row 2: cumulative RMSE
    fig.add_trace(go.Scatter(x=time[:-1], y=rmse_c[:-1], line=dict(color='#ef4444', dash='dot', width=2), showlegend=False), row=2, col=2)
    fig.add_trace(go.Scatter(x=time[:-1], y=rmse_h[:-1], line=dict(color='#10b981', dash='dot', width=3), showlegend=False), row=2, col=2)

    # Right column, row 3: total accel-bias Norm tracking
    fig.add_trace(go.Scatter(x=time[:-1], y=np.full_like(time[:-1], true_ba_norm), name='True Norm', line=dict(color='black', dash='dash')), row=3, col=2)
    fig.add_trace(go.Scatter(x=time[:-1], y=ba_c_norm, line=dict(color='#ef4444', width=2), showlegend=False), row=3, col=2)
    fig.add_trace(go.Scatter(x=time[:-1], y=ba_h_norm, line=dict(color='#10b981', width=2), showlegend=False), row=3, col=2)

    # Left column, row 4: total gyro-drift Norm tracking
    fig.add_trace(go.Scatter(x=time[:-1], y=np.full_like(time[:-1], true_bg_norm), line=dict(color='black', dash='dash'), showlegend=False), row=4, col=1)
    fig.add_trace(go.Scatter(x=time[:-1], y=bg_c_norm, line=dict(color='#ef4444', width=2), showlegend=False), row=4, col=1)
    fig.add_trace(go.Scatter(x=time[:-1], y=bg_h_norm, line=dict(color='#10b981', width=2), showlegend=False), row=4, col=1)

    fig.update_layout(height=1100, title_text="Classic vs Hybrid: Navigation Performance",
                      template="plotly_white", hovermode="x unified")
    fig.update_xaxes(title_text="Time [s]", row=4, col=1)
    fig.show()


# Run and plot
(time, out_pos_c, out_az_err_c, out_pos_h, out_az_err_h, init_pos, test_bias, test_drift,
 out_ba_c, out_bg_c, out_ba_h, out_bg_h, out_nn_ba, out_nn_bg) = run_ab_test_with_zupt()

plot_final_ab_test(time, out_pos_c, out_az_err_c, out_pos_h, out_az_err_h, init_pos,
                   test_bias, test_drift, out_ba_c, out_bg_c, out_ba_h, out_bg_h)

LSTM weights loaded.
Running Classic vs Hybrid comparison (with ZUPT)...
